# Diabetes Hospital Readmission — Exploratory Data Analysis

This notebook contains the initial exploration of the data used in our NOVA IMS Data Science Bootcamp project, which aims to predict hospital readmission risk for diabetic patients.

We work with two files:
- `diabetic_data.csv`: the main dataset, with ~101,766 hospital encounters and 50 features (patient demographics, admission details, diagnoses, medications, and the readmission outcome).
- `IDS_mapping.csv`: a lookup table that translates the numeric codes used in `diabetic_data.csv` (admission type, discharge disposition, admission source) into their text descriptions. It is not a dataset on its own: it contains 3 small tables stacked on top of each other.

This section covers: loading the data with the correct missing-value handling, decoding the ID columns with the lookup table, a first look at the structure and content of the data (shape, preview, data types, descriptive statistics), and an automated profiling report for a more in-depth data quality assessment.

**Note on loading.** The raw data is always loaded through `src/data_prep.py`, never with a bare `pd.read_csv()`. With the pandas defaults, the string `"None"` (= "test not performed" in `max_glu_serum` and `A1Cresult`) would wrongly become NaN, while `"?"` (the real missing-value marker) would not.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from ydata_profiling import ProfileReport

# Find the project root, starting from the current working directory
current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir

# Make the shared code in src/ importable from the notebook
sys.path.insert(0, str(project_root / "src"))
from data_prep import ID_COLUMNS, add_mapping_descriptions, load_mappings, load_raw

/var/folders/f8/r14dd3cn66q9d1d3lmtz0lth0000gn/T/ipykernel_41956/1916040394.py:5: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


In [2]:
# Check which Python environment the notebook is running on
print(sys.executable)

/Users/catotix/Documents/Programações/Diabetes-Hospital-Readmission-Data-Science-Project/.venv/bin/python


In [3]:
# Load the main dataset ("?" -> NaN, "None" kept as a category, ID columns as category)
df_diabetes = load_raw()

# Parse the lookup table into one {id: description} dict per ID column
mappings = load_mappings()

# Add the decoded *_desc columns next to the original IDs
df_diabetes = add_mapping_descriptions(df_diabetes, mappings)

In [4]:
df_diabetes.shape

(101766, 53)

In [5]:
df_diabetes.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,admission_type_desc,discharge_disposition_desc,admission_source_desc
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,NO,Unknown,Unknown,Physician Referral
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,No,No,No,Ch,Yes,>30,Emergency,Discharged to home,Emergency Room
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,Yes,NO,Emergency,Discharged to home,Emergency Room
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,No,No,No,Ch,Yes,NO,Emergency,Discharged to home,Emergency Room
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,No,No,No,Ch,Yes,NO,Emergency,Discharged to home,Emergency Room


In [6]:
df_diabetes.describe(include="all")

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,admission_type_desc,discharge_disposition_desc,admission_source_desc
count,1.017660e+05,1.017660e+05,99493,101766,101766,3197,101766.0,101766.0,101766.0,101766.000000,...,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766
unique,NaN,NaN,5,3,10,9,8.0,26.0,17.0,NaN,...,2,2,2,2,2,2,3,6,25,15
top,NaN,NaN,Caucasian,Female,[70-80),[75-100),1.0,1.0,7.0,NaN,...,No,No,No,No,No,Yes,NO,Emergency,Discharged to home,Emergency Room
freq,NaN,NaN,76099,54708,26068,1336,53990.0,60234.0,57494.0,NaN,...,101753,101765,101764,101765,54755,78363,54864,53990,60234,57494
mean,1.652016e+08,5.433040e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.395987,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,1.026403e+08,3.869636e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.985108,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,1.252200e+04,1.350000e+02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,8.496119e+07,2.341322e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,1.523890e+08,4.550514e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,2.302709e+08,8.754595e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
df_diabetes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 53 columns):
 #   Column                      Non-Null Count   Dtype   
---  ------                      --------------   -----   
 0   encounter_id                101766 non-null  int64   
 1   patient_nbr                 101766 non-null  int64   
 2   race                        99493 non-null   object  
 3   gender                      101766 non-null  object  
 4   age                         101766 non-null  object  
 5   weight                      3197 non-null    object  
 6   admission_type_id           101766 non-null  category
 7   discharge_disposition_id    101766 non-null  category
 8   admission_source_id         101766 non-null  category
 9   time_in_hospital            101766 non-null  int64   
 10  payer_code                  61510 non-null   object  
 11  medical_specialty           51817 non-null   object  
 12  num_lab_procedures          101766 non-null  int64   
 13 

### The lookup table (IDS_mapping.csv)

The file is parsed into 3 dictionaries, one per ID column. Descriptions that carry no information (`NULL`, `Not Mapped`, `Not Available`, `Unknown/Invalid`) are merged into a single `Unknown` category in the `*_desc` columns.

In [8]:
# Number of codes described for each ID column
{col: len(mapping) for col, mapping in mappings.items()}

{'admission_type_id': 8,
 'discharge_disposition_id': 30,
 'admission_source_id': 25}

In [9]:
# Distribution of the decoded descriptions
for col in ID_COLUMNS:
    desc_col = col.replace("_id", "_desc")
    print(df_diabetes[desc_col].value_counts(), "\n")

admission_type_desc
Emergency        53990
Elective         18869
Urgent           18480
Unknown          10396
Trauma Center       21
Newborn             10
Name: count, dtype: int64 

discharge_disposition_desc
Discharged to home                                                                                           60234
Discharged/transferred to SNF                                                                                13954
Discharged/transferred to home with home health service                                                      12902
Unknown                                                                                                       4680
Discharged/transferred to another short term hospital                                                         2128
Discharged/transferred to another rehab fac including rehab units of a hospital .                             1993
Expired                                                                                          

## Data profiling with ydata-profiling

We generate an automated exploratory report for the main dataset, after the correct loading and with the ID columns cast to category.
The report includes: descriptive statistics per variable, distributions, missing values, duplicates, correlations, and automatic warnings about potential data quality issues.

The lookup table is not profiled: it is only used to decode the ID columns, so a report on it has no analytical value.

The report is saved to `reports/df_diabetes_report.html`, the only HTML report committed to Git, so it can be opened from GitHub without regenerating it.

In [10]:
# # Generate the profiling report for the main dataset (diabetic_data.csv)
# # NB: ~100,000 rows and 50+ columns, so this can take a few minutes to run
##profile_diabetes = ProfileReport(
##    df_diabetes, title="Diabetic Data - Profiling Report", explorative=True
##)

# # Save the report as an HTML file in the reports/ folder
##profile_diabetes.to_file(project_root / "reports" / "df_diabetes_report.html")

> The `df_diabetes_report.html` file in `reports/` can be opened directly in a browser to review the results, without having to re-run the code every time.